# Dimer MD Analysis — RMSD, RMSF, RoG, SASA (Final)
Analysis pipeline for ID protein dimers (ID1-ID1, ID4-ID1), using `.gro` + `.xtc` files.

**RMSF fix:** computed **per chain**, each chain aligned on itself only (not the
whole dimer), so inter-chain wobble doesn't inflate the fluctuation values.

**Chain detection:** this `.gro` has no usable `segid`/`chainID` (everything is
`'SYSTEM'`) and no bond connectivity, so chains are detected by finding where
the residue numbering **resets** (e.g. ...,154,155,1,2,...) in the CA atom
order. This was verified directly on this data: chain1 = residues 0-155,
chain2 = residues 155-310 (atom index ranges).

**Unified x-axis:** the two chains' RMSF values are concatenated into one
continuous line per system (chain 1 residues, then chain 2 right after),
matching how RMSD/RoG/SASA are already plotted as one line per system. A
dotted vertical line marks the chain1 -> chain2 boundary.

In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis import rms, align
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
import warnings
import plotly.io as pio

warnings.filterwarnings("ignore")
pio.templates.default = "simple_white"

COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e"]


def update_fig(fig, titlename, xname, yname, y=0.9, font_size=22):
    fig.update_layout(
        title={"text": titlename, "y": y, "x": 0.5, "xanchor": "center", "yanchor": "top"},
        xaxis_title=xname, yaxis_title=yname,
        font_family="Times New Roman", font_size=font_size,
        xaxis=dict(showline=True, linecolor="black", mirror=True, ticks="outside"),
        yaxis=dict(showline=True, linecolor="black", mirror=True, ticks="outside"),
        legend=dict(y=-0.22, xanchor="center", x=0.5, orientation="h", borderwidth=1),
    )
    return fig


## Chain detection helper (resid-reset method)

Works for `.gro`-only files with no `segid`/`chainID`/bond info: detects a
new chain wherever the CA `resid` sequence drops back down (e.g. 155 -> 1).

In [ ]:
def get_chain_atomgroups(universe):
    """
    Split protein CA atoms into chains by detecting resid resets
    (e.g. ...,154,155,1,2,...). Returns a list of
    (label, "atom_range", (start_index, end_index)) tuples, where the
    indices are positions within the CA-only AtomGroup (not global atom
    indices).
    """
    ca = universe.select_atoms("protein and name CA")
    resids = ca.resids

    boundaries = [0]
    for j in range(1, len(resids)):
        if resids[j] < resids[j - 1]:   # resid dropped -> new chain starts here
            boundaries.append(j)
    boundaries.append(len(resids))

    chain_groups = []
    for idx in range(len(boundaries) - 1):
        start, end = boundaries[idx], boundaries[idx + 1]
        chain_groups.append((f"chain{idx+1}", "atom_range", (start, end)))
    return chain_groups


def chain_selection_string(universe, kind, key):
    """Build an MDAnalysis selection string for one chain's CA atoms."""
    if kind == "atom_range":
        ca = universe.select_atoms("protein and name CA")
        start, end = key
        indices = ca.indices[start:end]
        return "index " + " ".join(map(str, indices))
    elif kind == "segid":
        return f"protein and segid {key} and name CA"
    elif kind == "chainID":
        return f"protein and chainID {key} and name CA"
    else:
        return "protein and name CA"


## System list

Edit the `gro`/`xtc`/`sasa_file` paths to match your own files.

In [ ]:
proteins = [
    {"name": "ID1-ID1", "gro": "step5_production_ID1-ID1.gro", "xtc": "step5_production_noPBC_ID1-ID1.xtc", "sasa_file": "area_ID1-ID1.xvg"},
    {"name": "ID4-ID1", "gro": "step5_production_ID4-ID1.gro", "xtc": "step5_production_noPBC_ID4-ID1.xtc", "sasa_file": "area_ID4-ID1.xvg"},
]


## (Optional) Quick sanity check on chain detection

Run this first to confirm chain boundaries look right for each system
before running the full analysis below.

In [ ]:
for protein in proteins:
    u_check = mda.Universe(protein["gro"])
    groups = get_chain_atomgroups(u_check)
    print(protein["name"], "->", [(g[0], g[2]) for g in groups])


## Main analysis loop

For each system:
- **RMSD**: whole-complex backbone RMSD over time.
- **RMSF**: computed *per chain* (aligned on that chain only), concatenated
  into one continuous residue axis, plotted as a single line per system.
- **Radius of Gyration**: whole-complex Rg over time.
- **SASA**: loaded from pre-computed `.xvg` files.

In [ ]:
fig_rmsd = go.Figure()
fig_rog  = go.Figure()
fig_sasa = go.Figure()

# RMSF gets its OWN subplot per system, since each complex has a different
# chain-length split -- overlaying them on one axis makes the boundary
# markers ambiguous (you can't tell which line belongs to which complex).
fig_rmsf = make_subplots(
    rows=len(proteins), cols=1,
    shared_xaxes=False,
    subplot_titles=[p["name"] for p in proteins],
    vertical_spacing=0.12,
)

for i, protein in enumerate(proteins):
    print(f"\nProcessing {protein['name']} ...")
    u = mda.Universe(protein["gro"], protein["xtc"])

    # ---------------- 1. RMSD (whole complex, backbone CA) ----------------
    R_rmsd = rms.RMSD(u, u, select="name CA", ref_frame=0).run()
    fig_rmsd.add_trace(go.Scatter(
        x=R_rmsd.results.rmsd[:, 1] / 1000, y=R_rmsd.results.rmsd[:, 2],
        name=protein["name"], line=dict(color=COLORS[i])
    ))

    # ---------------- 2. RMSF (per chain, then unified x-axis, own subplot) ----------------
    u_probe = mda.Universe(protein["gro"], protein["xtc"])
    chain_groups = get_chain_atomgroups(u_probe)
    print(f"  Detected {len(chain_groups)} chain(s): {[c[0] for c in chain_groups]}")

    combined_x, combined_y, chain_boundaries = [], [], []
    running_offset = 0

    for label, kind, key in chain_groups:
        u_chain = mda.Universe(protein["gro"], protein["xtc"])  # fresh universe per chain
        sel_str = chain_selection_string(u_chain, kind, key)

        align.AlignTraj(u_chain, u_chain, select=sel_str, in_memory=True).run()
        c_alphas = u_chain.select_atoms(sel_str)
        R_rmsf = rms.RMSF(c_alphas).run()

        n_res = len(c_alphas)
        chain_boundaries.append(running_offset)
        combined_x.extend(range(running_offset + 1, running_offset + n_res + 1))
        combined_y.extend(R_rmsf.results.rmsf)
        running_offset += n_res
        print(f"    {label}: {n_res} residues, mean RMSF = {np.mean(R_rmsf.results.rmsf):.2f} A")

    row = i + 1
    fig_rmsf.add_trace(
        go.Scatter(x=combined_x, y=combined_y, name=protein["name"],
                   line=dict(color=COLORS[i]), showlegend=False),
        row=row, col=1
    )
    # Boundary marker: same color family as the trace it belongs to (darker/
    # thinner line), drawn only within this subplot's own axes -- so there's
    # no ambiguity about which complex it refers to.
    y_max = max(combined_y) * 1.1
    for b in chain_boundaries[1:]:
        fig_rmsf.add_shape(
            type="line",
            x0=b + 0.5, x1=b + 0.5,
            y0=0, y1=y_max,
            line=dict(color=COLORS[i], dash="dot", width=1.5),
            opacity=0.6,
            row=row, col=1,
        )
    fig_rmsf.update_xaxes(title_text="Residue position (chain 1 then chain 2)", row=row, col=1)
    fig_rmsf.update_yaxes(title_text="RMSF (\u00c5)", row=row, col=1)

    # ---------------- 3. Radius of Gyration ----------------
    prot_atoms = u.select_atoms("protein")
    rog_vals = [prot_atoms.radius_of_gyration() for ts in u.trajectory]
    time_ns = np.linspace(0, u.trajectory.totaltime / 1000, len(rog_vals))
    fig_rog.add_trace(go.Scatter(x=time_ns, y=rog_vals, name=protein["name"], line=dict(color=COLORS[i])))

    # ---------------- 4. SASA (precomputed .xvg) ----------------
    try:
        sasa_data = np.transpose(np.loadtxt(protein["sasa_file"], comments=["@", "#"]))
        fig_sasa.add_trace(go.Scatter(
            x=sasa_data[0] / 1000, y=sasa_data[1] * 100,
            name=protein["name"], line=dict(color=COLORS[i])
        ))
    except Exception:
        print(f"  Warning: SASA file {protein['sasa_file']} not found.")

fig_rmsf.update_layout(
    title={"text": "Per-Chain C\u03b1 RMSF (one subplot per complex)", "x": 0.5, "xanchor": "center"},
    font_family="Times New Roman", font_size=18,
    showlegend=False,
    height=350 * len(proteins),
)


## Save & display figures

In [ ]:
# RMSF figure is already fully laid out (subplots, titles, axis labels) above,
# so it does NOT go through update_fig() -- doing so would overwrite its
# per-subplot configuration.
fig_rmsf.write_html("RMSF_Comp.html")
fig_rmsf.show()

figs = [
    (fig_rmsd, "Protein RMSD", "Time (ns)", "RMSD (\u00c5)", "RMSD_Comp.html"),
    (fig_rog,  "Radius of Gyration", "Time (ns)", "RoG (\u00c5)", "RoG_Comp.html"),
    (fig_sasa, "SASA", "Time (ns)", "Area (\u00c5\u00b2)", "SASA_Comp.html"),
]

for f, title, x, y, filename in figs:
    f = update_fig(f, title, x, y)
    f.write_html(filename)
    f.show()
